# Late Fusion (TF-IDF+LR 확률 + BERT+MLP 확률 가중합)

- `P_final = alpha * P(LR) + (1 - alpha) * P(MLP)`
- alpha in {0.0, ..., 1.0}, **val에서 best alpha 선택**, test는 1회
- LR 파라미터: baseline 튜닝값 (C=1, ngram(1,1), max_features=10000, balanced)
- **선행:** `bert_mlp.ipynb` 실행해 `mlp_val_proba.npy`, `mlp_test_proba.npy` 저장돼 있어야 함

In [ ]:
import os, time, numpy as np, pandas as pd

# 스모크 테스트: True면 클래스별 소량 샘플만 사용 (파이프라인 점검용)
SMOKE_TEST = False
N_SMOKE = 1000

def find_root(start='.'):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'processed_data')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(start)

ROOT = find_root()
DATA_DIR = os.path.join(ROOT, 'processed_data')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(ART_DIR, exist_ok=True)
print('ROOT     :', ROOT)
print('DATA_DIR :', DATA_DIR)
print('ART_DIR  :', ART_DIR)

In [ ]:
# 라벨 순서 (label_mapping.csv 기준: 인덱스 = label 값)
LABEL_NAMES = ['가사', '개인정보/ICT', '근로자', '금융조세', '기업',
               '민사', '특허/저작권', '행정', '형사A(생활형)', '형사B(일반형)']
N_CLASS = 10

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
def load_split(name):
    """결합 입력(text = 판시사항 + 요약문)과 라벨 반환."""
    df = pd.read_csv(os.path.join(DATA_DIR, name + '.csv'))
    if SMOKE_TEST:
        per = max(2, N_SMOKE // N_CLASS)
        df = df.groupby('label', group_keys=False).apply(
            lambda d: d.sample(min(len(d), per), random_state=42))
    text = (df['jdgmn'].fillna('') + ' ' + df['summ_pass'].fillna('')).tolist()
    labels = df['label'].to_numpy()
    return text, labels

In [ ]:
tr_text, ytr = load_split('train')
val_text, yval = load_split('val')
te_text, yte = load_split('test')

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 1))
Xtr = tfidf.fit_transform(tr_text)
lr = LogisticRegression(C=1, class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(Xtr, ytr)
lr_val = lr.predict_proba(tfidf.transform(val_text))
lr_test = lr.predict_proba(tfidf.transform(te_text))
print('LR classes:', lr.classes_.tolist())

In [ ]:
mlp_val = np.load(os.path.join(ART_DIR, 'mlp_val_proba.npy'))
mlp_test = np.load(os.path.join(ART_DIR, 'mlp_test_proba.npy'))
assert lr_val.shape == mlp_val.shape, (lr_val.shape, mlp_val.shape)
print('shapes OK:', lr_val.shape, mlp_val.shape)

In [ ]:
alphas = np.linspace(0, 1, 11)
val_f1s = []
for a in alphas:
    p = a * lr_val + (1 - a) * mlp_val
    val_f1s.append(f1_score(yval, p.argmax(1), average='macro'))
best_i = int(np.argmax(val_f1s))
best_alpha = alphas[best_i]
print(f'best alpha = {best_alpha:.1f}  (val F1-macro {val_f1s[best_i]:.4f})')
for a, f in zip(alphas, val_f1s):
    print(f'  alpha={a:.1f}  val F1={f:.4f}')

In [ ]:
p_test = best_alpha * lr_test + (1 - best_alpha) * mlp_test
yp = p_test.argmax(1)
print(f'TEST (alpha={best_alpha:.1f})  Accuracy {accuracy_score(yte, yp):.4f}  |  F1-macro {f1_score(yte, yp, average="macro"):.4f}  (baseline 0.6347)')
print(classification_report(yte, yp, target_names=LABEL_NAMES, digits=4))

In [ ]:
# alpha vs val F1 그래프 (H3 해석용)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, val_f1s, marker='o')
ax.axvline(best_alpha, color='red', ls='--', label=f'best alpha={best_alpha:.1f}')
ax.set_xlabel('alpha (LR weight)'); ax.set_ylabel('val F1-macro')
ax.set_title('Late Fusion: alpha vs val F1-macro'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(ART_DIR, 'late_fusion_alpha.png'), dpi=150)
plt.show()